# 🌿 02 Training — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/02_training.ipynb`                |
| **Tujuan** | Bangun arsitektur LSTM, latih model dengan data training, simpan model & histori. |
| **Input**  | `artifacts/data_train.npz`, `artifacts/label_info.json` |
| **Output** | `artifacts/model_bonsai_lstm.keras`, `artifacts/training_history.csv` |
| **Urutan** | Jalankan setelah: `01_preprocessing.ipynb` |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : D:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── IMPORT LIBRARY, KONSTANTA, SEED ───────────────────────────────────
import random, os
import warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

ARTIFACTS_DIR  = "../artifacts"
MODEL_PATH     = f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras"
HISTORY_PATH   = f"{ARTIFACTS_DIR}/training_history.csv"

EPOCHS         = 50
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
VAL_SPLIT      = 0.10
PATIENCE       = 10

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("✅ Konfigurasi training siap.")

[SEED] Global random seed = 42
✅ Konfigurasi training siap.


In [3]:
# ── RULE-NB-06: Validasi Artefak ───────────────────────────────────────
REQUIRED_ARTIFACTS = [
    f"{ARTIFACTS_DIR}/data_train.npz",
    f"{ARTIFACTS_DIR}/label_info.json",
]
missing = [f for f in REQUIRED_ARTIFACTS if not os.path.exists(f)]
assert not missing, (
    f"⛔ Artefak tidak ditemukan: {missing}\n"
    "Jalankan notebook sebelumnya terlebih dahulu."
)
print("✅ Semua artefak yang dibutuhkan tersedia.")

✅ Semua artefak yang dibutuhkan tersedia.


In [4]:
# ── RULE-TRAIN-02: Muat Artefak dari Preprocessing ─────────────────────
import json

train_data  = np.load(f"{ARTIFACTS_DIR}/data_train.npz")
X_train     = train_data["X_train"]
y_train     = train_data["y_train_cls"]
y_train_reg = train_data["y_train_reg"]

with open(f"{ARTIFACTS_DIR}/label_info.json") as f:
    label_info = json.load(f)

LOOK_BACK  = label_info["look_back"]
N_FEATURES = label_info["n_features"]

print(f"[LOAD] X_train : {X_train.shape}")
print(f"[LOAD] y_train : {y_train.shape}  (0={( y_train==0).sum()}, 1={(y_train==1).sum()})")

[LOAD] X_train : (3432, 24, 3)
[LOAD] y_train : (3432,)  (0=3378, 1=54)


In [5]:
# ── RULE-TRAIN-03: Penanganan Class Imbalance (Oversampling) ────────────
from sklearn.utils import resample

X_train_0 = X_train[y_train == 0]
y_train_0 = y_train[y_train == 0]
y_train_reg_0 = y_train_reg[y_train == 0]

X_train_1 = X_train[y_train == 1]
y_train_1 = y_train[y_train == 1]
y_train_reg_1 = y_train_reg[y_train == 1]

# Oversample minority class (1) to match 20% of majority class instead of full 1:1
# to avoid excessive overfitting while still giving more signal.
X_train_1_os, y_train_1_os, y_train_reg_1_os = resample( # type: ignore
    X_train_1, y_train_1, y_train_reg_1,
    replace=True, n_samples=len(y_train_0) // 5, random_state=SEED
)

X_train_bal = np.vstack([X_train_0, X_train_1_os])
y_train_bal = np.hstack([y_train_0, y_train_1_os])
y_train_reg_bal = np.hstack([y_train_reg_0, y_train_reg_1_os])

# Shuffle balanced data
indices = np.arange(len(y_train_bal))
np.random.shuffle(indices)
X_train = X_train_bal[indices]
y_train = y_train_bal[indices]
y_train_reg = y_train_reg_bal[indices]

# Recalculate class weights after oversampling
classes  = np.unique(y_train)
cw_arr   = compute_class_weight("balanced", classes=classes, y=y_train)
cw_dict  = dict(zip(classes.tolist(), cw_arr.tolist()))

print(f"[OS] Setelah oversampling: 0={ (y_train==0).sum()}, 1={(y_train==1).sum()}")
print(f"[CW] Class weights baru: {cw_dict}")

[OS] Setelah oversampling: 0=3378, 1=675
[CW] Class weights baru: {0: 0.599911190053286, 1: 3.002222222222222}


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras import Model

def build_model(look_back: int, n_features: int) -> Model:
    inputs = Input(shape=(look_back, n_features))
    x = LSTM(64, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    
    # Head Klasifikasi
    cls_x = Dense(16, activation="relu")(x)
    output_cls = Dense(1, activation="sigmoid", name="classification")(cls_x)
    
    # Head Regresi (Prediksi soil moisture 0-1)
    reg_x = Dense(16, activation="relu")(x)
    output_reg = Dense(1, activation="linear", name="regression")(reg_x)
    
    model = Model(inputs=inputs, outputs=[output_cls, output_reg], name="bonsai_lstm")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={"classification": "binary_crossentropy", "regression": "mse"},
        metrics={"classification": ["accuracy", tf.keras.metrics.AUC(name="auc")], "regression": ["mae"]},
        loss_weights={"classification": 1.0, "regression": 20.0}, # Tingkatkan bobot regresi
    )
    return model

In [7]:
model = build_model(LOOK_BACK, N_FEATURES)
model.summary()

Model: "bonsai_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 24, 3)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 24, 64)            │          17,408 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 24, 64)            │               0 │ lstm[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ (None, 32)                │          12,416 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 32)                │               0 │ lstm_1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 16)                │             528 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 16)                │             528 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classification (Dense)        │ (None, 1)                 │              17 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ regression (Dense)            │ (None, 1)                 │              17 │ dense_1[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 30,914 (120.76 KB)

 Trainable params: 30,914 (120.76 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# ── RULE-TRAIN-05: Callbacks Wajib ──────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    CSVLogger(HISTORY_PATH, separator=",", append=False),
]
print("✅ Callbacks configured.")

✅ Callbacks configured.


In [9]:
# ── RULE-TRAIN-06: Eksekusi Pelatihan ──────────────────────────────────
# Normalisasi target regresi ke range 0-1 (bagi 100)
y_train_reg_scaled = (y_train_reg / 100.0).astype("float32")
y_train_cls_fix = y_train.reshape(-1, 1).astype("float32")
y_train_reg_fix = y_train_reg_scaled.reshape(-1, 1).astype("float32")

# Create sample weights for each head
sample_weights_cls = np.array([cw_dict[int(y)] for y in y_train], dtype="float32")
sample_weights_reg = np.ones_like(y_train, dtype="float32")

# Keras 3 expects sample_weight to have same structure as output labels (list)
sample_weight_list = [sample_weights_cls, sample_weights_reg]

history = model.fit(
    X_train, [y_train_cls_fix, y_train_reg_fix],
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = VAL_SPLIT,
    callbacks        = callbacks,
    sample_weight    = sample_weight_list, # type: ignore
    verbose          = 2,
)
print(f"[TRAIN] Model disimpan -> {MODEL_PATH}")
print(f"[TRAIN] Histori disimpan -> {HISTORY_PATH}")

Epoch 1/50



Epoch 1: val_loss improved from None to 0.69563, saving model to ../artifacts/model_bonsai_lstm.keras



Epoch 1: finished saving model to ../artifacts/model_bonsai_lstm.keras


114/114 - 5s - 41ms/step - classification_accuracy: 0.7568 - classification_auc: 0.7119 - classification_loss: 0.6648 - loss: 1.6517 - regression_loss: 0.0493 - regression_mae: 0.1426 - val_classification_accuracy: 0.8128 - val_classification_auc: 0.9993 - val_classification_loss: 0.6592 - val_loss: 0.6956 - val_regression_loss: 0.0020 - val_regression_mae: 0.0308


Epoch 2/50


...

Epoch 50/50



Epoch 50: val_loss did not improve from 0.01005


114/114 - 1s - 11ms/step - classification_accuracy: 0.9951 - classification_auc: 0.9991 - classification_loss: 0.0127 - loss: 0.0256 - regression_loss: 6.4518e-04 - regression_mae: 0.0166 - val_classification_accuracy: 0.9975 - val_classification_auc: 1.0000 - val_classification_loss: 0.0066 - val_loss: 0.0170 - val_regression_loss: 5.1024e-04 - val_regression_mae: 0.0106


Restoring model weights from the end of the best epoch: 48.


[TRAIN] Model disimpan -> ../artifacts/model_bonsai_lstm.keras
[TRAIN] Histori disimpan -> ../artifacts/training_history.csv


In [10]:
# ── RULE-TRAIN-07: Tampilkan Ringkasan Training ────────────────────────
hist_df    = pd.read_csv(HISTORY_PATH)
best_epoch = int(hist_df["val_loss"].idxmin()) + 1

print(f"\n[SUMMARY] Epoch terbaik    : {best_epoch} / {len(hist_df)}")
print(f"[SUMMARY] Val Loss terbaik : {hist_df['val_loss'].min():.4f}")
print(f"[SUMMARY] Val Accuracy     : {hist_df['val_classification_accuracy'].iloc[best_epoch-1]:.4f}")
print(f"[SUMMARY] Val AUC          : {hist_df['val_classification_auc'].iloc[best_epoch-1]:.4f}")


[SUMMARY] Epoch terbaik    : 48 / 50
[SUMMARY] Val Loss terbaik : 0.0100
[SUMMARY] Val Accuracy     : 1.0000
[SUMMARY] Val AUC          : 1.0000


## 📊 Ringkasan Training

**Model yang dihasilkan:**
- `model_bonsai_lstm.keras` → Model LSTM terbaik (berdasarkan val_loss minimum)
- `training_history.csv` → Histori loss, accuracy, AUC per epoch

**Langkah selanjutnya:**
Jalankan `notebooks/03_testing.ipynb` untuk menjalankan prediksi pada data testing.